# 04. Neural Collaborative Filtering (NCF)

논문: *He et al., "Neural Collaborative Filtering" (WWW 2017)*

## MF의 한계 → NCF의 동기

MF는 **내적(dot product)** 으로 유저-아이템 관계를 표현합니다.  
내적은 선형 연산이라 복잡한 비선형 상호작용을 표현하지 못합니다.

```
MF:  score(u, i) = uᵀ · i          ← 선형
NCF: score(u, i) = f(u, i; θ)      ← 신경망으로 임의의 함수 근사
```

## 아키텍처: NeuMF = GMF + MLP

```
user_id ──▶ GMF_user_emb ──▶ ⊙ (element-wise) ──────────────────────▶ ┐
item_id ──▶ GMF_item_emb ──▶ ⊙                                         │
                                                            concat ──▶ Linear(1)
user_id ──▶ MLP_user_emb ──▶ concat ──▶ FC─ReLU ──▶ FC─ReLU ──▶ ... ──▶ ┘
item_id ──▶ MLP_item_emb ──▶
```

- **GMF branch**: MF를 일반화. element-wise product → 선형 상호작용 포착  
- **MLP branch**: concat 후 FC 레이어 → 비선형 상호작용 포착  
- 두 브랜치가 **독립 임베딩** 을 사용하는 이유: 서로 다른 정보 학습

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from pathlib import Path

from models.mf import BPRDataset, MatrixFactorization
from models.ncf import NeuMF
from models.evaluate import evaluate

DATA_DIR = Path('../data/processed/All_Beauty')
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
test_df  = pd.read_parquet(DATA_DIR / 'test.parquet')
with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)
n_users, n_items = meta['n_users'], meta['n_items']

## 1. 모델 구조 비교

In [ ]:
N_FACTORS = 32
mf_model  = MatrixFactorization(n_users, n_items, N_FACTORS)
ncf_model = NeuMF(n_users, n_items, N_FACTORS, mlp_layers=[64, 32, 16])

mf_params  = sum(p.numel() for p in mf_model.parameters())
ncf_params = sum(p.numel() for p in ncf_model.parameters())

print(f"{'모델':<8} {'파라미터 수':>12} {'구조'}")
print("-" * 55)
print(f"{'MF':<8} {mf_params:>12,} user_emb({N_FACTORS}) + item_emb({N_FACTORS}) → dot")
print(f"{'NCF':<8} {ncf_params:>12,} GMF embs × 2 + MLP embs × 2 + FC layers")
print()

print("NCF 상세 구조:")
print(ncf_model)

## 2. GMF vs MLP — 각 브랜치가 보는 것

**GMF** (Generalized Matrix Factorization):
```python
gmf_out = gmf_user_emb[u] * gmf_item_emb[i]   # element-wise
# MF의 내적과 다름: 내적은 sum(gmf_out), GMF는 벡터 전체를 유지
# 최종 레이어가 각 차원의 중요도를 다르게 학습 가능
```

**MLP**:
```python
mlp_input = concat(mlp_user_emb[u], mlp_item_emb[i])  # (64,)
mlp_out   = ReLU(Linear(64→32)) → ReLU(Linear(32→16))
# concat: 두 벡터의 상호작용을 FC 레이어가 자유롭게 조합
# 내적처럼 차원별 곱이 아니라 모든 차원 조합 가능
```

In [ ]:
# 브랜치별 출력 텐서 shape 확인
dummy_users = torch.zeros(4, dtype=torch.long)
dummy_items = torch.tensor([0, 1, 2, 3], dtype=torch.long)

gmf_u = ncf_model.gmf_user_emb(dummy_users)
gmf_i = ncf_model.gmf_item_emb(dummy_items)
gmf_out = gmf_u * gmf_i

mlp_u = ncf_model.mlp_user_emb(dummy_users)
mlp_i = ncf_model.mlp_item_emb(dummy_items)
mlp_cat = torch.cat([mlp_u, mlp_i], dim=-1)
mlp_out = ncf_model.mlp(mlp_cat)

combined = torch.cat([gmf_out, mlp_out], dim=-1)
final    = ncf_model.predict_layer(combined)

print(f"입력 user/item: {dummy_users.shape}")
print(f"GMF output    : {gmf_out.shape}  ← element-wise product")
print(f"MLP input     : {mlp_cat.shape}  ← concat(user_emb, item_emb)")
print(f"MLP output    : {mlp_out.shape}")
print(f"NeuMF concat  : {combined.shape} ← GMF({N_FACTORS}) + MLP(16)")
print(f"최종 점수     : {final.shape}")

## 3. 학습 및 MF와 수렴 속도 비교

In [ ]:
def train_model(mdl, n_epochs=50, lr=1e-3, batch_size=512):
    dataset   = BPRDataset(train_df, n_items)
    loader    = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(mdl.parameters(), lr=lr)
    history   = []
    for epoch in range(1, n_epochs + 1):
        mdl.train()
        total_loss = 0.0
        for users, pos_items, neg_items in loader:
            optimizer.zero_grad()
            loss = mdl.bpr_loss(users, pos_items, neg_items)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        history.append({'epoch': epoch, 'loss': total_loss / len(loader)})
    return history

mf_model  = MatrixFactorization(n_users, n_items, N_FACTORS)
ncf_model = NeuMF(n_users, n_items, N_FACTORS, mlp_layers=[64, 32, 16])

print("MF 학습 중...")
mf_hist  = train_model(mf_model)
print("NCF 학습 중...")
ncf_hist = train_model(ncf_model)
print("완료")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
epochs = [h['epoch'] for h in mf_hist]
ax.plot(epochs, [h['loss'] for h in mf_hist],  label='MF',  color='steelblue')
ax.plot(epochs, [h['loss'] for h in ncf_hist], label='NCF', color='coral')
ax.set_xlabel('Epoch')
ax.set_ylabel('BPR Loss')
ax.set_title('MF vs NCF — 학습 손실 비교')
ax.legend()
plt.tight_layout()
plt.show()
print("NCF가 더 낮은 loss에 도달하지만 수렴이 느린 이유:")
print("  → 파라미터 수가 많아 최적화 공간이 더 복잡하기 때문")

## 4. 평가 비교: Hit@10, NDCG@10

In [ ]:
def get_score_fn(mdl):
    mdl.eval()
    def score_fn(user_id, item_ids):
        u = torch.tensor([user_id] * len(item_ids))
        i = torch.tensor(item_ids)
        with torch.no_grad():
            return mdl(u, i)
    return score_fn

mf_metrics  = evaluate(get_score_fn(mf_model),  train_df, test_df, n_items, K=10)
ncf_metrics = evaluate(get_score_fn(ncf_model), train_df, test_df, n_items, K=10)

comparison = pd.DataFrame([
    {'model': 'MF',  **mf_metrics,  'params': sum(p.numel() for p in mf_model.parameters())},
    {'model': 'NCF', **ncf_metrics, 'params': sum(p.numel() for p in ncf_model.parameters())},
]).set_index('model')

display(comparison)

hit_gain  = (ncf_metrics['hit@10']  - mf_metrics['hit@10'])  / mf_metrics['hit@10']  * 100
ndcg_gain = (ncf_metrics['ndcg@10'] - mf_metrics['ndcg@10']) / mf_metrics['ndcg@10'] * 100
print(f"\nNCF 개선율: Hit@10 {hit_gain:+.1f}%,  NDCG@10 {ndcg_gain:+.1f}%")

## 5. 결과 해석 — 왜 차이가 작은가?

이 데이터셋에서 MF와 NCF 성능 차이가 크지 않은 이유:

| 요인 | 설명 |
|------|------|
| **데이터 규모** | 2,087 users / 2,216 items는 MLP의 비선형 표현력이 충분히 발휘되기에 작음 |
| **희소성** | sparsity 99.7% — 관측된 신호가 너무 적어 복잡한 모델도 학습 어려움 |
| **시퀀스 정보 없음** | 유저의 행동 순서를 무시 → 다음 단계(SASRec)에서 개선 |

**논문 vs 실제:**  
NCF 논문에서 큰 개선이 나오는 건 대규모 데이터 (MovieLens 1M, Pinterest)에서입니다.  
이 소규모 데이터에서 작은 차이도 의미 있는 학습이 이뤄졌다는 증거입니다.

In [ ]:
# 유저별로 MF vs NCF 추천 결과 차이 확인
with open(DATA_DIR / 'item2id.json') as f:
    item2id = json.load(f)
id2item = {v: k for k, v in item2id.items()}
meta_df = pd.read_parquet(DATA_DIR / 'meta.parquet')[['parent_asin', 'title']].drop_duplicates('parent_asin')

def top_k_items(mdl, user_id, k=5):
    seen = set(train_df[train_df['user_id'] == user_id]['parent_asin'])
    scores = mdl.score_all_items(user_id).numpy()
    ranked = sorted(
        [(item_id, scores[item_id]) for item_id in range(n_items) if item_id not in seen],
        key=lambda x: -x[1]
    )[:k]
    return [
        str(meta_df[meta_df['parent_asin'] == id2item.get(iid, '')]['title'].values[0])[:45]
        if len(meta_df[meta_df['parent_asin'] == id2item.get(iid, '')]) > 0 else id2item.get(iid, '?')
        for iid, _ in ranked
    ]

user_id = 10
print(f"유저 {user_id}에 대한 추천 (Top-5)\n")
print(f"{'순위':<4} {'MF':<47} {'NCF'}")
print("-" * 100)
for rank, (mf_item, ncf_item) in enumerate(zip(top_k_items(mf_model, user_id), top_k_items(ncf_model, user_id)), 1):
    marker = "  ← 동일" if mf_item == ncf_item else ""
    print(f"{rank:<4} {mf_item:<47} {ncf_item}{marker}")

---
## 요약

| | MF | NCF |
|-|-|-|
| 상호작용 모델링 | 내적 (선형) | GMF + MLP (선형 + 비선형) |
| 임베딩 세트 | 1쌍 | 2쌍 (GMF용, MLP용) |
| 수렴 속도 | 빠름 | 느림 (파라미터 多) |
| 소규모 데이터 | 충분 | 과소적합 위험 |
| 대규모 데이터 | 표현력 한계 | 개선 효과 큼 |

**두 모델의 공통 한계:**  
유저의 전체 이력을 하나의 벡터로 압축 → **취향 변화 순서**를 무시합니다.  

→ **다음 스텝**: SASRec — Transformer로 이력의 순서를 직접 학습